In [1]:
import pandas as pd
import numpy as np
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler
from scipy.stats import mode
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

In [2]:
# ==============================================================================
# 1. LOAD RAW DATA
# ==============================================================================
print("Loading raw time series data...")
df_raw = pd.read_parquet('../data/raw/X_train.parquet').reset_index()
unique_ids = df_raw['id'].unique()

print(f"Total unique series to process: {len(unique_ids)}")

Loading raw time series data...
Total unique series to process: 10001


In [ ]:
# ==============================================================================
# 2. HMM FEATURE ENGINEERING (THE REGIME SCANNER)
# ==============================================================================
hmm_features = []
errors = 0

print("Training HMMs and extracting regime features...")
for uid in tqdm(unique_ids, desc="HMM Extraction"):
    
    # 1. Isolate and prepare the data
    series_data = df_raw[df_raw['id'] == uid].sort_values('time')
    
    pre_mask = series_data['period'] == 0
    post_mask = series_data['period'] == 1
    
    pre_vals = series_data[pre_mask]['value'].values.reshape(-1, 1)
    post_vals = series_data[post_mask]['value'].values.reshape(-1, 1)
    all_vals = series_data['value'].values.reshape(-1, 1)
    
    # Base dictionary
    feat_dict = {
        'id': uid, 'hmm_ll_diff': np.nan, 'hmm_ll_ratio': np.nan, 
        'hmm_post_ll_per_step': np.nan, 'hmm_viterbi_transition': 0, 
        'hmm_dominant_state_change': 0, 'hmm_state_purity_diff': np.nan
    }
    
    # Skip if data is too short for a reliable HMM
    if len(pre_vals) < 20 or len(post_vals) < 5:
        hmm_features.append(feat_dict)
        continue
        
    try:
        # CRITICAL: Scale the data. HMMs use Gaussian emissions and need scaled variance.
        scaler = StandardScaler()
        pre_scaled = scaler.fit_transform(pre_vals)
        # Transform post based on pre's scale to detect anomalies correctly
        post_scaled = scaler.transform(post_vals) 
        
        scaler_all = StandardScaler()
        all_scaled = scaler_all.fit_transform(all_vals)
        
        # ----------------------------------------------------------------------
        # OPTION A: Anomaly Detector (Trained ONLY on 'pre')
        # ----------------------------------------------------------------------
        model_A = hmm.GaussianHMM(n_components=2, covariance_type="diag", n_iter=50, random_state=42)
        model_A.fit(pre_scaled)
        
        ll_pre = model_A.score(pre_scaled)
        ll_post = model_A.score(post_scaled)
        
        # Normalize by length because LL is a cumulative sum
        norm_ll_pre = ll_pre / len(pre_scaled)
        norm_ll_post = ll_post / len(post_scaled)
        
        feat_dict['hmm_ll_diff'] = norm_ll_pre - norm_ll_post
        # Use abs() to avoid sign problems with negative numbers.
        feat_dict['hmm_ll_ratio'] = abs(norm_ll_post) / (abs(norm_ll_pre) + 1e-6) 
        feat_dict['hmm_post_ll_per_step'] = norm_ll_post
        
        # ----------------------------------------------------------------------
        # OPTION B: Regime Decoder (Trained on the FULL series)
        # ----------------------------------------------------------------------
        model_B = hmm.GaussianHMM(n_components=2, covariance_type="diag", n_iter=50, random_state=42)
        model_B.fit(all_scaled)
        
        # Viterbi Algorithm returns the most likely sequence of hidden states
        hidden_states = model_B.predict(all_scaled)
        
        # Split the path back into pre and post
        states_pre = hidden_states[:len(pre_scaled)]
        states_post = hidden_states[len(pre_scaled):]
        
        # Feature: Dominant state change
        mode_pre = mode(states_pre, keepdims=False).mode
        mode_post = mode(states_post, keepdims=False).mode
        feat_dict['hmm_dominant_state_change'] = 1 if mode_pre != mode_post else 0
        
        # Feature: Purity difference (Max percentage of time spent in a single state)
        purity_pre = np.max(np.bincount(states_pre)) / len(states_pre)
        purity_post = np.max(np.bincount(states_post)) / len(states_post)
        feat_dict['hmm_state_purity_diff'] = purity_pre - purity_post
        
        # Feature: Viterbi exact transition
        # Check if the state changed exactly around the breakpoint (last pre vs first post)
        if states_pre[-1] != states_post[0]:
            feat_dict['hmm_viterbi_transition'] = 1
            
    except Exception as e:
        errors += 1
        pass
        
    hmm_features.append(feat_dict)

print(f"\nExtraction complete. Failed series: {errors} / {len(unique_ids)}")

Training HMMs and extracting regime features...


HMM Extraction: 100%|██████████| 10001/10001 [34:02<00:00,  4.90it/s]


Extraction complete. Failed series: 0 / 10001


In [ ]:
# ==============================================================================
# 3. SAVE TO PARQUET
# ==============================================================================
df_hmm = pd.DataFrame(hmm_features).set_index('id')

df_hmm = df_hmm.fillna(df_hmm.median())

save_path = '../data/processed/features_hmm.parquet'
df_hmm.to_parquet(save_path)

print(f"HMM Features saved to '{save_path}'")
display(df_hmm.head())

HMM Features saved to '../data/processed/features_hmm.parquet'


,hmm_ll_diff,hmm_ll_ratio,hmm_post_ll_per_step,hmm_viterbi_transition,hmm_dominant_state_change,hmm_state_purity_diff
id,,,,,,
0,-0.014009,0.990094,-1.400313,1,0,-0.016637
1,-0.163952,0.873089,-1.127921,0,0,-0.255736
2,0.278030,1.207754,-1.616290,0,0,0.267279
3,0.091871,1.067207,-1.458840,0,0,0.069469
4,0.028052,1.019769,-1.447009,0,0,0.000000
